# Concept Agent Test (Stage 9)

Full pipeline:
1. Load ChromaDB index (retriever)
2. Retrieve relevant chunks for a question
3. Extract structured concepts via `ConceptAgent`

Each concept includes:
- **concept** — short name
- **definition** — grounded definition from the context
- **relevance** — why it matters for the research focus

**Prerequisites:**
- Index already created (`01_test_rag_pipeline.ipynb`, Part A)
- `.env` with `HUGGINGFACE_API_TOKEN` (Stage 7)
- `uv sync`

## Configuration

In [9]:
import json
import os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VECTOR_DB_PATH = ROOT / "data/vector_db"

TOP_K = 5
QUESTION = "What are the core concepts behind the Transformer architecture?"

token = os.getenv("HUGGINGFACE_API_TOKEN") or os.getenv("HF_TOKEN")

print(f"Project root: {ROOT}")
print(f"Vector DB: {VECTOR_DB_PATH}")
print(f"Default question: {QUESTION}")
print(f"HF token: {'configured' if token else 'NOT configured'}")

Project root: d:\Github\research-workflow-agent
Vector DB: d:\Github\research-workflow-agent\data\vector_db
Default question: What are the core concepts behind the Transformer architecture?
HF token: configured


## 1. Load retriever

In [10]:
from rag.embeddings import ChunkEmbedder
from rag.retriever import DocumentRetriever
from rag.vectorstore import ChromaVectorStore

store = ChromaVectorStore(persist_directory=VECTOR_DB_PATH)
store.load_collection()

chunk_count = store.collection.count()
print(f"Collection: {store.collection_name} ({chunk_count} chunks)")

if chunk_count == 0:
    raise RuntimeError(
        "Empty index. Run notebook 01_test_rag_pipeline (Part A) first."
    )

embedder = ChunkEmbedder()
retriever = DocumentRetriever(vector_store=store, embedder=embedder, top_k=TOP_K)

Collection: book_research (52 chunks)


## 2. Retrieve context (RAG)

In [11]:
context = retriever.invoke(QUESTION)

print(f"Question: {QUESTION}")
print(f"Chunks retrieved: {len(context)}")
print()

for i, doc in enumerate(context, start=1):
    print(f"--- Chunk {i} | page={doc.metadata.get('page')} | distance={doc.metadata.get('distance')} ---")
    print(doc.page_content[:250].replace("\n", " "))
    print()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2963.10it/s]


Question: What are the core concepts behind the Transformer architecture?
Chunks retrieved: 5

--- Chunk 1 | page=2 | distance=0.27702444791793823 ---
Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, re

--- Chunk 2 | page=1 | distance=0.35985565185546875 ---
In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelizat

--- Chunk 3 | page=0 | distance=0.37867307662963867 ---
best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data. ∗Equal contribution. Listing order i

## 3. Run Concept Agent

In [12]:
from agents.concepts_agent import ConceptAgent

print("Calling Concept Agent (Hugging Face API)...")
agent = ConceptAgent()
result = agent.run(QUESTION, context)

Calling Concept Agent (Hugging Face API)...


In [13]:
print("Structured JSON output:")
print(result.to_json())

Structured JSON output:
{
  "concepts": [
    {
      "concept": "Transformer Architecture",
      "definition": "A neural network model that relies entirely on self-attention to draw global dependencies between input and output, replacing recurrence and convolutional neural networks.",
      "relevance": "The Transformer architecture is the main focus of the research, and its design allows for more parallelization and better performance in translation quality."
    },
    {
      "concept": "Self-Attention",
      "definition": "A mechanism that allows each position in a sequence to attend to all positions in the sequence, drawing global dependencies between input and output.",
      "relevance": "Self-attention is a key component of the Transformer architecture, enabling it to compute representations of its input and output without using sequence-aligned RNNs or convolution."
    },
    {
      "concept": "Encoder-Decoder Structure",
      "definition": "A neural network architecture

In [14]:
print("=" * 60)
print("EXTRACTED CONCEPTS")
print("=" * 60)

for i, item in enumerate(result.concepts, start=1):
    print(f"\n{i}. {item.concept}")
    print(f"   Definition: {item.definition}")
    print(f"   Relevance:  {item.relevance}")

EXTRACTED CONCEPTS

1. Transformer Architecture
   Definition: A neural network model that relies entirely on self-attention to draw global dependencies between input and output, replacing recurrence and convolutional neural networks.
   Relevance:  The Transformer architecture is the main focus of the research, and its design allows for more parallelization and better performance in translation quality.

2. Self-Attention
   Definition: A mechanism that allows each position in a sequence to attend to all positions in the sequence, drawing global dependencies between input and output.
   Relevance:  Self-attention is a key component of the Transformer architecture, enabling it to compute representations of its input and output without using sequence-aligned RNNs or convolution.

3. Encoder-Decoder Structure
   Definition: A neural network architecture consisting of an encoder that maps input sequences to continuous representations and a decoder that generates output sequences based on 

## 4. Test another question

Change `another_question` and run the cells below.

In [15]:
another_question = "What role does attention play in sequence modeling?"

context2 = retriever.invoke(another_question)
result2 = agent.run(another_question, context2)

print(f"Question: {another_question}")
print(json.dumps(result2.to_dict(), indent=2, ensure_ascii=False))

Question: What role does attention play in sequence modeling?
{
  "concepts": [
    {
      "concept": "Attention Mechanism",
      "definition": "A mechanism allowing modeling of dependencies without regard to their distance in the input or output sequences.",
      "relevance": "Critical for sequence modeling and transduction models, enabling the modeling of long-range dependencies."
    },
    {
      "concept": "Self-Attention",
      "definition": "An attention mechanism relating different positions of a single sequence to compute a representation of the sequence.",
      "relevance": "Used in various tasks, including reading comprehension and abstractive summarization, due to its ability to model long-range dependencies."
    },
    {
      "concept": "Transformer",
      "definition": "A model architecture relying entirely on an attention mechanism to draw global dependencies between input and output sequences, dispensing with recurrence and convolutions.",
      "relevance": "P